# Speaker study: Phase 1 for Qwen 2.5 7B base (primary model) + Phase 2 token checks

**Part A (Phase 1):** reproduces the paper's 4.1 screen for `Qwen/Qwen2.5-7B` (bf16, layer 8),
using the same script that passed on Gemma 2 2B. Qwen 7B does not fit on a free T4 in bf16,
so the model is built with only its first 9 decoder blocks (`--truncate`). The readout at block 8
does not depend on later blocks, and the Gemma run verified this empirically (0.0 difference on
21 items). Here, reproducing the shipped values, which the authors computed with the full model,
is itself the test.

**Part B (Phase 2, tokenizers only, no forward passes):** builds the label-swapped stimuli
(`[Assistant]:` -> `[User]:` / `[Moderator]:`) and checks, for the Gemma and Qwen tokenizers, that
the token sequences differ only within the final label, and which final token each condition ends on.

**How to run:** T4 GPU runtime, then Runtime -> Run all. About 15-20 min, mostly the ~15 GB download.
The last cell downloads `speaker_study_upload.zip` (also saved to `MyDrive/speaker_study/`);
attach it in the Claude session.

In [ ]:
# 1. GPU check. Runtime -> Change runtime type -> T4 GPU (free tier).
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: switch the runtime to a GPU first."

In [ ]:
# 2. Mount Google Drive; all outputs go to MyDrive/speaker_study/ (never overwritten).
from google.colab import drive
drive.mount("/content/drive")
OUT_ROOT = "/content/drive/MyDrive/speaker_study"
import os; os.makedirs(OUT_ROOT, exist_ok=True)

In [ ]:
# 3. Hugging Face token from Colab secrets (key icon on the left, name HF_TOKEN, notebook access on).
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

In [ ]:
# 4. Clone the paper's repo at the pinned commit (read-only use; nothing in it is modified).
!rm -rf /content/Pain-axis
!git clone -q https://github.com/valen-research/Pain-axis /content/Pain-axis
!git -C /content/Pain-axis checkout -q 7c256502ed3d98e4e6379290fe7db2f93cb8d025
!git -C /content/Pain-axis log -1 --format='%H %s'
# Uses Colab's preinstalled torch/transformers/accelerate/pandas (versions are logged to env.txt).
!mkdir -p /content/speaker_study_scripts

In [ ]:
%%writefile /content/speaker_study_scripts/pa_common.py
"""Shared helpers for the speaker study: dataset and vector loading, model loading, and
final-token readout. The format, readout and projection logic is copied from
Pain-axis scripts/4.1_self_other/01_screen_scenarios.py (commit 7c25650) so that our
projections are comparable with the shipped ones.
"""

import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch

PAIN_AXIS_COMMIT = "7c256502ed3d98e4e6379290fe7db2f93cb8d025"
SEED = 0

# Dataset stratum -> our group name.
STRATUM_TO_GROUP = {
    "self_directed": "harm_to_model",
    "vicarious_empathic": "user_suffering",
    "neutral_filler": "neutral",
}

# Same keys and order as the repo's 4.1 screen.
VECTOR_KEYS = [
    "s1_pain_vector", "s2_pain_vector",
    "fear_vector", "negemotion_vector", "negworld_vector",
    "bodysens_vector", "arousal_vector", "random_vector", "numb_vector", "sadness_vector",
]

# Base models from the spec: HF repo -> repo's model name (used in vector/result file names).
MODELS = {
    "google/gemma-2-2b": "Gemma_2_2B_base",
    "Qwen/Qwen2.5-7B": "Qwen_2.5_7B_base",
    "meta-llama/Llama-3.1-8B": "Llama_3.1_8B_base",
    "google/gemma-2-9b": "Gemma_2_9B_base",
}

DTYPES = {"bf16": torch.bfloat16, "fp16": torch.float16, "fp32": torch.float32}


def load_scenarios(pain_axis_dir):
    path = Path(pain_axis_dir) / "datasets" / "4.1_self_other_420_scenarios.json"
    with open(path, encoding="utf-8") as f:
        items = json.load(f)
    for it in items:
        it["group"] = STRATUM_TO_GROUP[it["stratum"]]
        it["n_user_turns"] = sum(line.startswith("[User]:") for line in it["text"].split("\n"))
    return items


def load_vectors(pain_axis_dir, model_name):
    """Unit vectors from the file the 4.1 screen uses, and its layer."""
    path = Path(pain_axis_dir) / "results" / "vectors_full_steering" / f"vectors_full_{model_name}.pt"
    data = torch.load(path, map_location="cpu", weights_only=False)
    units = {}
    for k in VECTOR_KEYS:
        if data.get(k) is not None:
            v = data[k].float().numpy()
            n = np.linalg.norm(v)
            units[k] = v / n if n > 0 else v
    return int(data["layer"]), units


def load_shipped_screen(pain_axis_dir, model_name):
    import pandas as pd
    path = Path(pain_axis_dir) / "results" / "4.1_self_other" / "per_model" / f"screen_v2_{model_name}.csv"
    return pd.read_csv(path)


def load_model(repo, dtype="bf16", attn="default", keep_layers=None):
    """Load tokenizer and model on the GPU.

    keep_layers: if set, build the model with only its first `keep_layers` decoder blocks.
    The output of block L depends only on blocks 0..L, so the readout is unchanged; this
    lets 7-8B models fit on a 16 GB GPU in bf16. Phase 1 verifies the equivalence.
    """
    import transformers
    from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(repo)
    kwargs = {"low_cpu_mem_usage": True, "device_map": "cuda" if torch.cuda.is_available() else "cpu"}
    # Newer transformers renamed torch_dtype -> dtype; an unknown kwarg would silently be
    # written into the config, so pick by version and assert the dtype afterwards.
    major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
    kwargs["dtype" if (major, minor) >= (4, 56) else "torch_dtype"] = DTYPES[dtype]
    if attn != "default":
        kwargs["attn_implementation"] = attn
    if keep_layers is not None:
        cfg = AutoConfig.from_pretrained(repo)
        cfg.num_hidden_layers = keep_layers
        if isinstance(getattr(cfg, "layer_types", None), list):
            cfg.layer_types = cfg.layer_types[:keep_layers]
        kwargs["config"] = cfg
    model = AutoModelForCausalLM.from_pretrained(repo, **kwargs)
    model.eval()
    got = next(model.parameters()).dtype
    assert got == DTYPES[dtype], f"model loaded as {got}, expected {DTYPES[dtype]}"
    return tok, model


def decoder_layers(model):
    return model.model.layers if hasattr(model.model, "layers") else model.model.language_model.layers


def encode(tok, text, device):
    """As the repo does for base models: default special tokens (adds BOS where the tokenizer does)."""
    return tok(text, return_tensors="pt").input_ids.to(device)


class FinalTokenReader:
    """Forward hook on decoder block `layer`; captures the final-token output in fp32."""

    def __init__(self, model, layer):
        self.model, self.act = model, None
        self.handle = decoder_layers(model)[layer].register_forward_hook(self._hook)

    def _hook(self, module, inputs, output):
        hs = output[0] if isinstance(output, tuple) else output
        self.act = hs[0, -1, :].float().cpu().numpy()

    @torch.no_grad()
    def __call__(self, input_ids):
        self.model(input_ids=input_ids)
        return self.act

    def remove(self):
        self.handle.remove()


def git_head(path):
    try:
        return subprocess.run(["git", "-C", str(path), "rev-parse", "HEAD"],
                              capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return "unknown"


def env_info():
    import transformers
    info = {"python": sys.version.split()[0], "torch": torch.__version__,
            "transformers": transformers.__version__, "cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
    if torch.cuda.is_available():
        info["gpu_capability"] = ".".join(map(str, torch.cuda.get_device_capability(0)))
    return info


def pip_freeze(path):
    out = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
    Path(path).write_text(out)


def zscore(x, mean, sd):
    return (np.asarray(x) - mean) / (sd + 1e-8)

In [ ]:
%%writefile /content/speaker_study_scripts/01_reproduce_4p1.py
"""Phase 1: reproduce the paper's Section 4.1 screen for one base model.

Runs the 420 scenarios in the original format ('...\\n[Assistant]:'), reads the final-token
output of the decoder block stored in the vector file, projects onto every unit
direction, z-scores against the 420-item pool (population SD, as the repo does), and
compares with the shipped screen: item-level raw projections and the 21 category means of
the pain axis (mean of S1 and S2 z).

Pass criterion (spec): category-mean r >= 0.95 and mean |diff| <= 0.10 z.

Also checks:
  - BOS at position 0 and the final token of every item;
  - the hook output equals hidden_states[layer + 1] (layer convention);
  - a model truncated to layer + 1 blocks gives the same activations (used later to fit
    7-8B models on a T4).

Writes to <out-root>/results/<model>/phase1_<run-id>/ and refuses to overwrite.

Example:
  python 01_reproduce_4p1.py --pain-axis-dir /content/Pain-axis \\
      --out-root /content/drive/MyDrive/speaker_study --model-repo google/gemma-2-2b
"""

import argparse
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import pa_common as pc

R_MIN, MAD_MAX = 0.95, 0.10


def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--pain-axis-dir", required=True, help="clone of valen-research/Pain-axis")
    p.add_argument("--out-root", required=True, help="speaker_study output root (e.g. on Drive)")
    p.add_argument("--model-repo", default="google/gemma-2-2b", choices=sorted(pc.MODELS))
    p.add_argument("--dtype", default="bf16", choices=sorted(pc.DTYPES))
    p.add_argument("--attn", default="default", choices=["default", "eager", "sdpa"])
    p.add_argument("--truncate", action="store_true",
                   help="run the main pass on the truncated model (needed for 7-8B on a T4)")
    p.add_argument("--n-trunc-check", type=int, default=21,
                   help="items for the truncation-equivalence check (0 to skip)")
    p.add_argument("--run-id", default=None, help="default: date_time_gpu")
    return p.parse_args()


def main():
    args = parse_args()
    np.random.seed(pc.SEED)
    torch.manual_seed(pc.SEED)
    model_name = pc.MODELS[args.model_repo]
    env = pc.env_info()
    run_id = args.run_id or time.strftime("%Y%m%d_%H%M%S") + "_" + (env["gpu"] or "cpu").replace(" ", "")
    out = Path(args.out_root) / "results" / model_name / f"phase1_{run_id}"
    if out.exists():
        raise SystemExit(f"{out} exists; refusing to overwrite")
    out.mkdir(parents=True)

    pa_commit = pc.git_head(args.pain_axis_dir)
    items = pc.load_scenarios(args.pain_axis_dir)
    layer, units = pc.load_vectors(args.pain_axis_dir, model_name)
    shipped = pc.load_shipped_screen(args.pain_axis_dir, model_name)
    print(f"{model_name}: layer {layer}, {len(units)} vectors, {len(items)} items; env {env}")
    if pa_commit != pc.PAIN_AXIS_COMMIT:
        print(f"WARNING: Pain-axis at {pa_commit}, study pinned to {pc.PAIN_AXIS_COMMIT}")
    pc.pip_freeze(out / "env.txt")

    # ---- main pass -------------------------------------------------------------------
    keep = layer + 1 if args.truncate else None
    tok, model = pc.load_model(args.model_repo, args.dtype, args.attn, keep_layers=keep)
    attn_impl = getattr(model.config, "_attn_implementation", "unknown")
    n_blocks = len(pc.decoder_layers(model))
    print(f"loaded: {n_blocks} blocks, dtype {next(model.parameters()).dtype}, attn {attn_impl}")
    first = pc.encode(tok, items[0]["text"], "cpu")[0]
    print("first item renders as:", repr(tok.decode(first)))
    reader = pc.FinalTokenReader(model, layer)

    # Layer convention: the hook on block `layer` must equal hidden_states[layer + 1].
    # When block `layer` is the model's last block (truncated model), HF returns that entry
    # after the final norm, so the hook output is normed before comparing.
    hs_check = []
    for it in items[:3]:
        ids = pc.encode(tok, it["text"], model.device)
        with torch.no_grad():
            hs = model(input_ids=ids, output_hidden_states=True).hidden_states
            a_hook = reader.act
            if layer + 1 == len(hs) - 1:
                h = torch.tensor(a_hook, device=model.device, dtype=next(model.parameters()).dtype)
                a_hook = model.model.norm(h[None, None])[0, 0].float().cpu().numpy()
        a_hs = hs[layer + 1][0, -1].float().cpu().numpy()
        a_prev = hs[layer][0, -1].float().cpu().numpy()
        hs_check.append({"id": it["id"], "max_abs_diff_vs_hs[L+1]": float(np.abs(a_hook - a_hs).max()),
                         "max_abs_diff_vs_hs[L]": float(np.abs(a_hook - a_prev).max())})
    print("layer-convention check:", hs_check)

    rows, acts = [], {}
    t0 = time.time()
    for i, it in enumerate(items):
        ids = pc.encode(tok, it["text"], model.device)
        act = reader(ids)
        acts[it["id"]] = act
        row = {"model": model_name, "scenario_id": it["id"], "category": it["category"],
               "group": it["group"], "n_user_turns": it["n_user_turns"], "n_tokens": ids.shape[1],
               "bos_first": bool(tok.bos_token_id is not None and ids[0, 0].item() == tok.bos_token_id),
               "final_token_id": int(ids[0, -1]), "final_token": tok.decode(ids[0, -1:]),
               "act_norm": float(np.linalg.norm(act))}
        for k, u in units.items():
            row[f"{k}_proj"] = float(np.dot(act, u))
        rows.append(row)
        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(items)}  ({time.time() - t0:.0f}s)")
    reader.remove()
    del model
    torch.cuda.empty_cache()

    df = pd.DataFrame(rows)
    pool = {}
    for k in units:
        m, s = float(df[f"{k}_proj"].mean()), float(df[f"{k}_proj"].std(ddof=0))
        pool[k] = {"mean": m, "sd": s}
        df[f"{k}_z"] = pc.zscore(df[f"{k}_proj"], m, s)
    df["pain_axis_z"] = (df["s1_pain_vector_z"] + df["s2_pain_vector_z"]) / 2
    df.to_csv(out / "items.csv", index=False)

    # ---- comparison with the shipped screen -----------------------------------------
    sh = shipped.rename(columns={"id": "scenario_id"}).copy()
    sh["pain_axis_z"] = (sh["s1_pain_vector_z"] + sh["s2_pain_vector_z"]) / 2
    mg = df.merge(sh, on="scenario_id", suffixes=("", "_shipped"))
    assert len(mg) == len(df) == 420, len(mg)

    item_level = {}
    for k in units:
        a, b = mg[f"{k}_proj"], mg[f"{k}_proj_shipped"]
        item_level[k] = {"r": float(np.corrcoef(a, b)[0, 1]), "mean_abs_diff": float((a - b).abs().mean()),
                         "max_abs_diff": float((a - b).abs().max())}
    a, b = mg["pain_axis_z"], mg["pain_axis_z_shipped"]
    item_level["pain_axis_z"] = {"r": float(np.corrcoef(a, b)[0, 1]), "mean_abs_diff": float((a - b).abs().mean()),
                                 "max_abs_diff": float((a - b).abs().max())}

    cat = mg.groupby(["group", "category"])[["pain_axis_z", "pain_axis_z_shipped"]].mean()
    cat["diff"] = cat["pain_axis_z"] - cat["pain_axis_z_shipped"]
    cat = cat.sort_values("pain_axis_z_shipped", ascending=False).round(3)
    r_cat = float(np.corrcoef(cat["pain_axis_z"], cat["pain_axis_z_shipped"])[0, 1])
    mad_cat = float(cat["diff"].abs().mean())
    passed = r_cat >= R_MIN and mad_cat <= MAD_MAX
    cat.to_csv(out / "category_means_vs_shipped.csv")

    other_axes = {}
    for k in ["s1_pain_vector", "s2_pain_vector", "fear_vector", "negemotion_vector", "sadness_vector"]:
        c = mg.groupby("category")[[f"{k}_z", f"{k}_z_shipped"]].mean()
        other_axes[k] = {"r": float(np.corrcoef(c.iloc[:, 0], c.iloc[:, 1])[0, 1]),
                         "mad": float((c.iloc[:, 0] - c.iloc[:, 1]).abs().mean())}

    # ---- truncation equivalence -------------------------------------------------------
    trunc = None
    if args.n_trunc_check > 0:
        sel = [it for j, it in enumerate(items) if j % max(1, len(items) // args.n_trunc_check) == 0][:args.n_trunc_check]
        other_keep = None if args.truncate else layer + 1
        tok2, model2 = pc.load_model(args.model_repo, args.dtype, args.attn, keep_layers=other_keep)
        reader2 = pc.FinalTokenReader(model2, layer)
        diffs = [float(np.abs(reader2(pc.encode(tok2, it["text"], model2.device)) - acts[it["id"]]).max()) for it in sel]
        reader2.remove()
        trunc = {"n_items": len(sel), "compared": "full vs truncated", "blocks_other_model": len(pc.decoder_layers(model2)),
                 "max_abs_diff": max(diffs), "all_exact": all(d == 0.0 for d in diffs)}
        del model2
        torch.cuda.empty_cache()
        print("truncation check:", trunc)

    final_tokens = df.groupby(["final_token_id", "final_token"]).size().reset_index(name="n")
    summary = {
        "phase": 1, "model": model_name, "model_repo": args.model_repo, "run_id": run_id,
        "pain_axis_commit": pa_commit, "layer": layer, "dtype": args.dtype, "attn_arg": args.attn,
        "attn_implementation": attn_impl, "truncated_main_pass": args.truncate, "env": env,
        "criterion": {"r_min": R_MIN, "mad_max": MAD_MAX},
        "category_means": {"r": r_cat, "mean_abs_diff": mad_cat, "max_abs_diff": float(cat["diff"].abs().max())},
        "PASS": passed,
        "item_level_vs_shipped": item_level,
        "category_means_other_axes": other_axes,
        "bos_first_all": bool(df["bos_first"].all()),
        "final_tokens": final_tokens.to_dict(orient="records"),
        "layer_convention_check": hs_check,
        "truncation_check": trunc,
        "pool_stats_assistant_next": pool,
    }
    (out / "summary.json").write_text(json.dumps(summary, indent=2))

    lines = [f"# Phase 1 reproduction: {model_name}, run {run_id}", "",
             f"GPU {env['gpu']}, torch {env['torch']}, transformers {env['transformers']}, "
             f"dtype {args.dtype}, attn {attn_impl}, layer {layer}", "",
             f"**Category means (21), pain axis z: r = {r_cat:.4f}, mean |diff| = {mad_cat:.4f} z "
             f"-> {'PASS' if passed else 'FAIL'}** (criterion r >= {R_MIN}, MAD <= {MAD_MAX})", "",
             "| group | category | ours | shipped | diff |", "|---|---|---|---|---|"]
    for (g, c), r in cat.iterrows():
        lines.append(f"| {g} | {c} | {r['pain_axis_z']:+.3f} | {r['pain_axis_z_shipped']:+.3f} | {r['diff']:+.3f} |")
    lines += ["", "Item-level raw projections vs shipped:", "",
              "| vector | r | mean abs diff | max abs diff |", "|---|---|---|---|"]
    for k, v in item_level.items():
        lines.append(f"| {k} | {v['r']:.5f} | {v['mean_abs_diff']:.4f} | {v['max_abs_diff']:.4f} |")
    lines += ["", f"BOS first on all items: {summary['bos_first_all']}",
              f"Final tokens: {summary['final_tokens']}",
              f"Layer convention: {hs_check}", f"Truncation: {trunc}"]
    (out / "REPORT_phase1.md").write_text("\n".join(lines) + "\n")
    print("\n".join(lines))
    print(f"\nwrote {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/speaker_study_scripts/02_build_stimuli.py
"""Phase 2: build the label-swapped stimuli and verify them.

For each of the 420 base-model transcripts, writes three variants that are byte-identical
except for the final speaker label:
  assistant_next   ...\\n[Assistant]:   (original)
  user_next        ...\\n[User]:
  moderator_next   ...\\n[Moderator]:   (third label, fixed before any Phase 3 run)

Text checks (always): each variant equals the shared prefix plus its label, the prefix
ends with '\\n', and the variants differ only after the prefix.

Token checks (with --tokenizers, needs Hugging Face access): tokenizes every variant,
finds the common token prefix across the three conditions, and records the differing
tail tokens and the final token of each condition. The tail may be shorter than the label
(e.g. if '\\n[' is one token shared by all labels); an item is flagged only if its tail
is not a non-empty suffix of the label, i.e. if tokens outside the label differ.

Outputs (never overwritten):
  <out-dir>/stimuli.jsonl                       one row per scenario x condition
  <out-dir>/token_check_<model>.csv / .json     per-tokenizer checks (if --tokenizers)
"""

import argparse
import json
from pathlib import Path

import pa_common as pc

LABELS = {"assistant_next": "[Assistant]:", "user_next": "[User]:", "moderator_next": "[Moderator]:"}
ORIG = LABELS["assistant_next"]


def build(items):
    rows = []
    for it in items:
        text = it["text"]
        assert text.endswith("\n" + ORIG), it["id"]
        prefix = text[: -len(ORIG)]
        for cond, label in LABELS.items():
            v = prefix + label
            assert v[: len(prefix)] == prefix and v[len(prefix):] == label
            rows.append({"scenario_id": it["id"], "category": it["category"], "group": it["group"],
                         "n_user_turns": it["n_user_turns"], "condition": cond, "label": label, "text": v})
        assert rows[-3]["text"] == text  # assistant_next is the original, byte for byte
    return rows


def token_check(rows, repo):
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(repo)
    by_id = {}
    for r in rows:
        by_id.setdefault(r["scenario_id"], {})[r["condition"]] = r
    out, problems = [], []
    for sid, conds in by_id.items():
        ids = {c: tok(conds[c]["text"]).input_ids for c in LABELS}
        n = min(len(v) for v in ids.values())
        k = 0
        while k < n and len({tuple(v[: k + 1]) for v in ids.values()}) == 1:
            k += 1
        rec = {"scenario_id": sid, "n_common_tokens": k}
        for c, v in ids.items():
            tail = v[k:]
            rec[f"{c}_n_tokens"] = len(v)
            rec[f"{c}_tail_ids"] = " ".join(map(str, tail))
            rec[f"{c}_tail"] = tok.decode(tail)
            rec[f"{c}_final_id"] = v[-1]
            rec[f"{c}_final"] = tok.decode(v[-1:])
            if not tail or not LABELS[c].endswith(tok.decode(tail)):
                problems.append((sid, c, tok.decode(tail)))
        out.append(rec)
    return out, problems


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--pain-axis-dir", required=True)
    p.add_argument("--out-dir", required=True)
    p.add_argument("--tokenizers", nargs="*", default=[], choices=sorted(pc.MODELS))
    args = p.parse_args()

    out = Path(args.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    rows = build(pc.load_scenarios(args.pain_axis_dir))
    stim = out / "stimuli.jsonl"
    if stim.exists():
        existing = [json.loads(line) for line in stim.read_text().splitlines()]
        assert existing == rows, f"{stim} exists with different content; refusing to overwrite"
        print(f"{stim} already exists and matches")
    else:
        stim.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows))
        print(f"wrote {stim}: {len(rows)} rows")

    import pandas as pd
    for repo in args.tokenizers:
        name = pc.MODELS[repo]
        csv_path, json_path = out / f"token_check_{name}.csv", out / f"token_check_{name}.json"
        if csv_path.exists():
            print(f"{csv_path} exists; skipping")
            continue
        recs, problems = token_check(rows, repo)
        df = pd.DataFrame(recs)
        df.to_csv(csv_path, index=False)
        summary = {"model": name, "repo": repo, "n_scenarios": len(df),
                   "n_tail_mismatches": len(problems), "tail_mismatch_examples": problems[:10]}
        for c in LABELS:
            summary[f"{c}_tails"] = df[f"{c}_tail_ids"].value_counts().to_dict()
            summary[f"{c}_final_tokens"] = (df[f"{c}_final"] + " (" + df[f"{c}_final_id"].astype(str) + ")").value_counts().to_dict()
        summary["same_final_token_all_conditions"] = bool(
            (df["assistant_next_final_id"] == df["user_next_final_id"]).all()
            and (df["assistant_next_final_id"] == df["moderator_next_final_id"]).all())
        json_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False))
        print(json.dumps(summary, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()

In [ ]:
# A. Phase 1 reproduction, Qwen 2.5 7B base (truncated to 9 blocks; the full-model comparison is skipped
#    because the full model does not fit on a T4).
%cd /content/speaker_study_scripts
!python 01_reproduce_4p1.py --pain-axis-dir /content/Pain-axis --out-root "{OUT_ROOT}" --model-repo Qwen/Qwen2.5-7B --dtype bf16 --truncate --n-trunc-check 0

In [ ]:
# B. Phase 2 stimuli + tokenizer checks (no model forward passes).
!python 02_build_stimuli.py --pain-axis-dir /content/Pain-axis --out-dir "{OUT_ROOT}/stimuli" --tokenizers google/gemma-2-2b Qwen/Qwen2.5-7B

In [ ]:
# C. Bundle the outputs for Claude: latest Qwen and Gemma Phase 1 runs + token checks.
import glob, os, zipfile
paths = []
for model in ["Qwen_2.5_7B_base", "Gemma_2_2B_base"]:
    runs = sorted(glob.glob(f"{OUT_ROOT}/results/{model}/phase1_*"))
    if runs:
        paths += glob.glob(runs[-1] + "/*")
paths += glob.glob(f"{OUT_ROOT}/stimuli/token_check_*")
zip_path = f"{OUT_ROOT}/speaker_study_upload.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        z.write(p, os.path.relpath(p, OUT_ROOT))
print("\n".join(os.path.relpath(p, OUT_ROOT) for p in paths))
from google.colab import files
files.download(zip_path)